In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import PowerTransformer
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_pickle('Data/student_dropout_bayes.pkl')

X = data.drop('Target', axis=1)
y = data['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Rozmiar zbioru treningowego (X_train): {X_train.shape}")
print(f"Rozmiar zbioru testowego (X_test): {X_test.shape}")

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', PowerTransformer(method='yeo-johnson', standardize=True)),
    ('classifier', GaussianNB())
])

In [ ]:
print("--- 5-Fold Cross-Validation ---")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')

print(f"Wyniki na poszczególnych podziałach: {cv_scores}")
print(f"Średnia dokładność: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

In [ ]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

y_pred_proba = pipeline.predict_proba(X_test)

print("--- RAPORT KLASYFIKACJI ---")
print(classification_report(y_test, y_pred))

print("\n--- METRYKA ROC AUC ---")
try:
    auc_score = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
    print(f"Wartość ROC AUC (Wieloklasowe OVR): {auc_score:.4f}")
except ValueError as e:
    print(f"Error: {e}")

In [ ]:
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=pipeline.classes_,
            yticklabels=pipeline.classes_)

plt.title('Macierz błędów (Confusion Matrix)')
plt.xlabel('Przewidziana klasa')
plt.ylabel('Rzeczywista klasa')
plt.show()